# Experiment 3: Classification using Deep Feed Forward Network and Logistic Regression

**Objective**: Implement classification using:
- a. Deep Feed Forward Network
- b. Logistic Regression

**Dataset**: Heart Failure Prediction Dataset (Kaggle)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

# Load environment variables
load_dotenv()

print(f"TensorFlow version: {tf.__version__}")

## 2. Download Dataset from Kaggle

In [ ]:
# Setup Kaggle API
import subprocess

# Create kaggle directory if not exists
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

# Download dataset
dataset_path = '../data/heart.csv'

if not os.path.exists(dataset_path):
    print("Downloading Heart Failure Prediction dataset from Kaggle...")
    try:
        subprocess.run([
            'kaggle', 'datasets', 'download', '-d', 'fedesoriano/heart-failure-prediction',
            '-p', '../data', '--unzip'
        ], check=True, capture_output=True)
        print("Dataset downloaded successfully!")
    except Exception as e:
        print(f"Error downloading: {e}")
        print("Using sklearn's breast cancer dataset as fallback...")
        from sklearn.datasets import load_breast_cancer
        data = load_breast_cancer()
        df = pd.DataFrame(data.data, columns=data.feature_names)
        df['target'] = data.target
        df.to_csv(dataset_path, index=False)
else:
    print("Dataset already exists.")

## 3. Load and Explore Dataset

In [ ]:
# Load dataset
df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Dataset info
print("\nDataset Info:")
df.info()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Target distribution
target_col = df.columns[-1]  # Assume last column is target
print(f"\nTarget Column: {target_col}")
print(f"Target Distribution:\n{df[target_col].value_counts()}")

## 4. Data Preprocessing

In [ ]:
# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove target from lists
if target_col in categorical_cols:
    categorical_cols.remove(target_col)
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

In [ ]:
# Encode categorical variables using Label Encoding
df_encoded = df.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f"Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nEncoded DataFrame head:")
df_encoded.head()

In [ ]:
# Separate features and target
X = df_encoded.drop(columns=[target_col]).values
y = df_encoded[target_col].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Training class distribution: {np.bincount(y_train)}")
print(f"Test class distribution: {np.bincount(y_test)}")

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied successfully.")
print(f"Scaled mean (train): {X_train_scaled.mean(axis=0).round(4)[:5]}... (first 5)")
print(f"Scaled std (train): {X_train_scaled.std(axis=0).round(4)[:5]}... (first 5)")

## 5. Data Visualization

In [ ]:
# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
df[target_col].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Target Class Distribution', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Correlation heatmap (numerical features only)
corr_data = df_encoded[numerical_cols + [target_col]].corr()
sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', ax=axes[1], square=True)
axes[1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/classification_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 6a. Logistic Regression

In [ ]:
# Train Logistic Regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)
y_prob_lr_test = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation metrics
lr_metrics = {
    'train_acc': accuracy_score(y_train, y_pred_lr_train),
    'test_acc': accuracy_score(y_test, y_pred_lr_test),
    'precision': precision_score(y_test, y_pred_lr_test),
    'recall': recall_score(y_test, y_pred_lr_test),
    'f1': f1_score(y_test, y_pred_lr_test)
}

print("Logistic Regression Results:")
print("="*40)
print(f"Training Accuracy: {lr_metrics['train_acc']:.4f}")
print(f"Test Accuracy: {lr_metrics['test_acc']:.4f}")
print(f"Precision: {lr_metrics['precision']:.4f}")
print(f"Recall: {lr_metrics['recall']:.4f}")
print(f"F1-Score: {lr_metrics['f1']:.4f}")

In [ ]:
# Classification Report
print("\nLogistic Regression - Classification Report:")
print(classification_report(y_test, y_pred_lr_test))

## 6b. Deep Feed Forward Network

In [ ]:
# Build Deep Feed Forward Network for classification
def build_dnn_classifier(input_shape):
    model = keras.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')  # Binary classification
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create model
dnn_model = build_dnn_classifier(X_train_scaled.shape[1])
dnn_model.summary()

In [ ]:
# Train DNN model
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

history = dnn_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Evaluate DNN model
y_prob_dnn_train = dnn_model.predict(X_train_scaled, verbose=0).flatten()
y_prob_dnn_test = dnn_model.predict(X_test_scaled, verbose=0).flatten()
y_pred_dnn_train = (y_prob_dnn_train >= 0.5).astype(int)
y_pred_dnn_test = (y_prob_dnn_test >= 0.5).astype(int)

dnn_metrics = {
    'train_acc': accuracy_score(y_train, y_pred_dnn_train),
    'test_acc': accuracy_score(y_test, y_pred_dnn_test),
    'precision': precision_score(y_test, y_pred_dnn_test),
    'recall': recall_score(y_test, y_pred_dnn_test),
    'f1': f1_score(y_test, y_pred_dnn_test)
}

print("\nDeep Feed Forward Network Results:")
print("="*40)
print(f"Training Accuracy: {dnn_metrics['train_acc']:.4f}")
print(f"Test Accuracy: {dnn_metrics['test_acc']:.4f}")
print(f"Precision: {dnn_metrics['precision']:.4f}")
print(f"Recall: {dnn_metrics['recall']:.4f}")
print(f"F1-Score: {dnn_metrics['f1']:.4f}")

In [ ]:
# Classification Report
print("\nDNN - Classification Report:")
print(classification_report(y_test, y_pred_dnn_test))

## 7. Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('DNN Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('DNN Training Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/classification_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Comparison

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr_test)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Logistic Regression\nAccuracy: {lr_metrics["test_acc"]:.4f}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# DNN
cm_dnn = confusion_matrix(y_test, y_pred_dnn_test)
sns.heatmap(cm_dnn, annot=True, fmt='d', cmap='Oranges', ax=axes[1])
axes[1].set_title(f'Deep Feed Forward Network\nAccuracy: {dnn_metrics["test_acc"]:.4f}', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../data/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))

# Logistic Regression ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr_test)
roc_auc_lr = auc(fpr_lr, tpr_lr)

# DNN ROC
fpr_dnn, tpr_dnn, _ = roc_curve(y_test, y_prob_dnn_test)
roc_auc_dnn = auc(fpr_dnn, tpr_dnn)

ax.plot(fpr_lr, tpr_lr, 'b-', linewidth=2, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})')
ax.plot(fpr_dnn, tpr_dnn, 'orange', linewidth=2, label=f'DNN (AUC = {roc_auc_dnn:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
# Comparison table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy (Train)', 'Accuracy (Test)', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'],
    'Logistic Regression': [
        lr_metrics['train_acc'], lr_metrics['test_acc'],
        lr_metrics['precision'], lr_metrics['recall'],
        lr_metrics['f1'], roc_auc_lr
    ],
    'Deep Feed Forward Network': [
        dnn_metrics['train_acc'], dnn_metrics['test_acc'],
        dnn_metrics['precision'], dnn_metrics['recall'],
        dnn_metrics['f1'], roc_auc_dnn
    ]
})

print("\n" + "="*60)
print("EXPERIMENT 3 SUMMARY: Classification")
print("="*60)
print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

winner = "Deep Feed Forward Network" if dnn_metrics['test_acc'] > lr_metrics['test_acc'] else "Logistic Regression"
print(f"\nBest Model: {winner}")

print("\nKey Observations:")
print("- Logistic Regression: Fast, interpretable, works well with linear boundaries")
print("- DNN: Can capture complex non-linear patterns, requires more data")
print("- Both models benefit from preprocessing (scaling, encoding)")
print("- AUC-ROC provides threshold-independent comparison")